# 🎯 Nexora — Unsupervised Customer Segmentation (K-Means)
**Phase 6: Behavioral Clustering & Persona Profiling**

### Objective
Discover natural customer segments across the 206,209 customer base using multidimensional RFM & behavioral features:
- Standardized feature transformation (`StandardScaler`).
- Mathematical selection of optimal $k$ using the **Elbow Method (Inertia)** and **Silhouette Scores**.
- Empirical statistical profiling of clusters without pre-assumptions.
- Actionable business persona mapping for marketing, retention, and personalization strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="deep")
DATA_DIR = Path("../data/processed")

df_cust = pd.read_parquet(DATA_DIR / "customer_features.parquet")
print(f"Loaded {len(df_cust):,} customers for segmentation analysis.")

---
## ❓ Business Question 1: What is the mathematically optimal number of customer clusters ($k$)?
*Compare Elbow Curve (Inertia) and Silhouette Scores across $k \in [2, 7]$ to select the optimal cluster partition.*

In [ ]:
features = [
    "user_total_orders",
    "user_avg_order_interval",
    "user_avg_basket_size",
    "user_reorder_rate",
    "user_unique_departments",
    "user_days_since_last_order"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cust[features])

# Subsample for fast silhouette computation
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=20000, replace=False)
X_sub = X_scaled[sample_idx]

k_range = range(2, 8)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sub, km.predict(X_sub)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
axes[0].plot(list(k_range), inertias, marker='o', color='#1d3557', linewidth=2)
axes[0].set_title("Elbow Method: Inertia vs. Number of Clusters (k)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (Sum of Squared Distances)")
axes[0].axvline(x=4, color='red', linestyle='--', label='Elbow at k=4')
axes[0].legend()

# Silhouette Score
axes[1].plot(list(k_range), silhouettes, marker='s', color='#e63946', linewidth=2)
axes[1].set_title("Silhouette Score vs. Number of Clusters (k)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Coefficient")
axes[1].axvline(x=4, color='red', linestyle='--', label='Optimal Separation (k=4)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## ❓ Business Question 2: What are the distinct statistical profiles and personas of each segment?
*Fit $k=4$ K-Means and evaluate empirical cluster centroids on unscaled features.*

In [ ]:
df_segments = pd.read_parquet(DATA_DIR / "customer_segments.parquet")
df_summary = pd.read_csv(DATA_DIR / "cluster_profile_summary.csv")

print("Cluster Profile Summary:")
display(df_summary)

# Radar / Bar comparison of segments
plt.figure(figsize=(12, 5))
sns.barplot(data=df_summary, x="segment_name", y="customer_count", palette="mako")
plt.title("Customer Segment Size Distribution", fontsize=14, fontweight="bold")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Users")
plt.xticks(rotation=15)
plt.show()